# 12 - Matching ottimizzato con CLIP, famiglie da JSON e foto vincitori

Questo notebook usa CLIP nel matching senza modificare i notebook precedenti.

Caratteristiche:

- le famiglie vengono caricate da un file JSON;
- il sesso viene usato come filtro;
- prima viene calcolato un ranking veloce strutturato;
- BERT e CLIP vengono calcolati solo sui Top 100 candidati;
- BERT e CLIP hanno lo stesso peso nella formula finale;
- per ogni famiglia vengono mostrate **tutte le foto del cane vincitore**.

Formula finale:

```text
final_score =
0.6 * structured_score +
0.2 * bert_similarity +
0.2 * clip_image_text_similarity
```

Regole razza:

```text
razza principale dataset  -> 1.0
razza secondaria dataset  -> 0.8
razza calcolata con CLIP  -> 0.5
```

La componente razza mantiene peso 3 nello `structured_score`.


## 1. Import

In [ ]:
from pathlib import Path
import json

import pandas as pd
import numpy as np
import torch
from PIL import Image
from IPython.display import display
from tqdm.auto import tqdm

from transformers import AutoTokenizer, AutoModel
from transformers import CLIPProcessor, CLIPModel
from sklearn.metrics.pairwise import cosine_similarity


## 2. Caricamento dati

In [ ]:
dogs = pd.read_csv("../data/processed/dogs_clean.csv")
bert_embeddings = np.load("../data/processed/bert_embeddings.npy")

clip_breeds_path = Path("../data/results/clip_breed_predictions_all_dogs.csv")
images_dir = Path("../data/raw/train_images")
results_dir = Path("../data/results")
results_dir.mkdir(parents=True, exist_ok=True)

if not clip_breeds_path.exists():
    raise FileNotFoundError(
        "File CLIP breeds non trovato. Prima esegui il notebook 11: "
        "../data/results/clip_breed_predictions_all_dogs.csv"
    )

clip_breeds = pd.read_csv(clip_breeds_path)

dogs = dogs.merge(
    clip_breeds[[
        "PetID",
        "clip_breed_1", "clip_breed_1_score",
        "clip_breed_2", "clip_breed_2_score",
        "clip_breed_3", "clip_breed_3_score",
        "clip_breed_4", "clip_breed_4_score",
        "clip_breed_5", "clip_breed_5_score",
        "num_images_used"
    ]],
    on="PetID",
    how="left"
)

for i in range(1, 6):
    dogs[f"clip_breed_{i}"] = dogs[f"clip_breed_{i}"].fillna("")
    dogs[f"clip_breed_{i}_score"] = dogs[f"clip_breed_{i}_score"].fillna(0.0)

dogs["num_images_used"] = dogs["num_images_used"].fillna(0)

print("Dataset cani:", dogs.shape)
print("Embedding BERT:", bert_embeddings.shape)
print("Predizioni razza CLIP:", clip_breeds.shape)
print("Cartella immagini esiste:", images_dir.exists())


## 3. Caricamento famiglie da file JSON

In [ ]:
possible_family_json_paths = [
    Path("../data/processed/family_profiles.json"),
    Path("../data/raw/family_profiles.json"),
    Path("../data/family_profiles.json"),
    Path("../family_profiles.json"),
    Path("family_profiles.json"),
]

family_json_path = None

for path in possible_family_json_paths:
    if path.exists():
        family_json_path = path
        break

if family_json_path is None:
    raise FileNotFoundError(
        "File JSON delle famiglie non trovato. "
        "Metti il file in una di queste posizioni oppure modifica family_json_path:\\n"
        + "\\n".join(str(p) for p in possible_family_json_paths)
    )

print("File famiglie trovato:", family_json_path)

with open(family_json_path, "r", encoding="utf-8") as f:
    raw_profiles = json.load(f)

if isinstance(raw_profiles, dict):
    family_profiles = raw_profiles
elif isinstance(raw_profiles, list):
    family_profiles = {}
    for i, profile in enumerate(raw_profiles, start=1):
        profile_id = profile.get("profile_id", profile.get("id", f"profile_{i}"))
        family_profiles[profile_id] = profile
else:
    raise ValueError("Formato JSON non valido: usa un dizionario o una lista di profili.")

print("Numero profili famiglia caricati:", len(family_profiles))
display(pd.DataFrame(family_profiles).T)


## 4. Normalizzazione dei profili famiglia

In [ ]:
def normalize_family_profile(profile):
    normalized = {
        "profile_name": profile.get("profile_name", profile.get("name", "Profilo famiglia")),
        "applicant_age": profile.get("applicant_age", "31-60"),
        "children": profile.get("children", False),
        "dog_experience": profile.get("dog_experience", "some"),
        "housing": profile.get("housing", "apartment"),
        "garden": profile.get("garden", False),
        "preferred_gender": profile.get("preferred_gender", "indifferent"),
        "preferred_age": profile.get("preferred_age", "indifferent"),
        "preferred_size": profile.get("preferred_size", "indifferent"),
        "preferred_fur": profile.get("preferred_fur", "indifferent"),
        "preferred_breeds": profile.get("preferred_breeds", []),
        "daily_time": profile.get("daily_time", "2-4"),
        "activity_level": profile.get("activity_level", "moderate"),
        "ideal_dog_description": profile.get(
            "ideal_dog_description",
            profile.get("description", "A suitable dog for this family.")
        )
    }

    if isinstance(normalized["preferred_breeds"], str):
        if normalized["preferred_breeds"].strip() == "":
            normalized["preferred_breeds"] = []
        else:
            normalized["preferred_breeds"] = [normalized["preferred_breeds"]]

    return normalized


family_profiles = {
    profile_id: normalize_family_profile(profile)
    for profile_id, profile in family_profiles.items()
}

display(pd.DataFrame(family_profiles).T)


## 5. Caricamento BERT e CLIP

In [ ]:
device = "cuda" if torch.cuda.is_available() else "cpu"

bert_model_name = "bert-base-uncased"
bert_tokenizer = AutoTokenizer.from_pretrained(bert_model_name)
bert_model = AutoModel.from_pretrained(bert_model_name).to(device)
bert_model.eval()

clip_model_name = "openai/clip-vit-base-patch32"
clip_model = CLIPModel.from_pretrained(clip_model_name).to(device)
clip_processor = CLIPProcessor.from_pretrained(clip_model_name)
clip_model.eval()

print("Device:", device)
print("BERT:", bert_model_name)
print("CLIP:", clip_model_name)


## 6. Funzioni BERT e CLIP

In [ ]:
def get_bert_embedding(text):
    inputs = bert_tokenizer(
        str(text),
        return_tensors="pt",
        truncation=True,
        padding=True,
        max_length=128
    ).to(device)

    with torch.no_grad():
        outputs = bert_model(**inputs)

    return outputs.last_hidden_state.mean(dim=1).squeeze().detach().cpu().numpy()


def get_clip_text_feature(text):
    with torch.no_grad():
        text_inputs = clip_processor(
            text=[str(text)],
            return_tensors="pt",
            padding=True,
            truncation=True
        ).to(device)

        text_outputs = clip_model.text_model(
            input_ids=text_inputs["input_ids"],
            attention_mask=text_inputs["attention_mask"]
        )

        text_feature = clip_model.text_projection(text_outputs.pooler_output)
        text_feature = text_feature / text_feature.norm(dim=-1, keepdim=True)

    return text_feature


def get_image_paths_for_pet(pet_id):
    pet_id = str(pet_id)

    paths = []
    paths.extend(sorted(images_dir.glob(f"{pet_id}-*.jpg")))
    paths.extend(sorted(images_dir.glob(f"{pet_id}-*.jpeg")))
    paths.extend(sorted(images_dir.glob(f"{pet_id}-*.png")))

    return sorted(set(paths))


_clip_image_feature_cache = {}

def get_clip_image_feature(image_path):
    image_path = str(image_path)

    if image_path in _clip_image_feature_cache:
        return _clip_image_feature_cache[image_path]

    image = Image.open(image_path).convert("RGB")

    with torch.no_grad():
        image_inputs = clip_processor(
            images=image,
            return_tensors="pt"
        ).to(device)

        image_outputs = clip_model.vision_model(
            pixel_values=image_inputs["pixel_values"]
        )

        image_feature = clip_model.visual_projection(image_outputs.pooler_output)
        image_feature = image_feature / image_feature.norm(dim=-1, keepdim=True)

    _clip_image_feature_cache[image_path] = image_feature
    return image_feature


_clip_pet_mean_feature_cache = {}

def get_clip_mean_image_feature_for_pet(pet_id):
    pet_id = str(pet_id)

    if pet_id in _clip_pet_mean_feature_cache:
        return _clip_pet_mean_feature_cache[pet_id]

    image_paths = get_image_paths_for_pet(pet_id)

    if len(image_paths) == 0:
        _clip_pet_mean_feature_cache[pet_id] = (None, 0)
        return None, 0

    features = []

    for image_path in image_paths:
        try:
            features.append(get_clip_image_feature(image_path))
        except Exception as e:
            print("Errore immagine:", image_path, "|", e)

    if len(features) == 0:
        _clip_pet_mean_feature_cache[pet_id] = (None, 0)
        return None, 0

    mean_feature = torch.stack(features).mean(dim=0)
    mean_feature = mean_feature / mean_feature.norm(dim=-1, keepdim=True)

    _clip_pet_mean_feature_cache[pet_id] = (mean_feature, len(features))
    return mean_feature, len(features)


## 7. Funzioni per mostrare tutte le foto del vincitore

In [ ]:
def display_all_dog_images(pet_id, max_width=260):
    image_paths = get_image_paths_for_pet(pet_id)

    if len(image_paths) == 0:
        print("Nessuna immagine disponibile per PetID:", pet_id)
        return

    print(f"Foto disponibili per {pet_id}: {len(image_paths)}")

    for image_path in image_paths:
        print(image_path.name)
        img = Image.open(image_path).convert("RGB")
        img.thumbnail((max_width, max_width))
        display(img)


## 8. Funzioni di scoring e filtri

In [ ]:
def normalize_text(text):
    return str(text).strip().lower()


def apply_hard_filters(data, family):
    filtered = data.copy()

    if family["daily_time"] == "<1":
        return filtered.iloc[0:0].copy(), "Meno di un'ora al giorno disponibile: nessun cane consigliato."

    if family["preferred_gender"] != "indifferent":
        filtered = filtered[filtered["gender_label"] == family["preferred_gender"]].copy()

    if not family["garden"]:
        filtered = filtered[filtered["maturity_size_label"] != "extra_large"].copy()

    if family["dog_experience"] == "none":
        filtered = filtered[filtered["maturity_size_label"] != "extra_large"].copy()

    return filtered.reset_index(drop=False).rename(columns={"index": "original_index"}), ""


def match_preference(preference, dog_value):
    if preference == "indifferent":
        return 1.0
    return 1.0 if preference == dog_value else 0.0


def size_similarity(preferred_size, dog_size):
    if preferred_size == "indifferent":
        return 1.0

    size_order = {"small": 0, "medium": 1, "large": 2, "extra_large": 3}

    if preferred_size not in size_order or dog_size not in size_order:
        return 0.0

    distance = abs(size_order[preferred_size] - size_order[dog_size])

    if distance == 0:
        return 1.0
    elif distance == 1:
        return 0.5
    else:
        return 0.0


def compute_breed_score(dog, family):
    preferred_breeds = family.get("preferred_breeds", [])

    if len(preferred_breeds) == 0:
        return None, "nessuna preferenza di razza: criterio ignorato"

    preferred_breeds = [normalize_text(breed) for breed in preferred_breeds]

    breed1 = normalize_text(dog.get("breed1_label", ""))
    breed2 = normalize_text(dog.get("breed2_label", ""))

    if breed1 in preferred_breeds:
        return 1.0, "razza principale del dataset corrisponde: +1.0"

    if breed2 in preferred_breeds:
        return 0.8, "razza secondaria del dataset corrisponde: +0.8"

    for i in range(1, 6):
        clip_breed = normalize_text(dog.get(f"clip_breed_{i}", ""))
        clip_score = float(dog.get(f"clip_breed_{i}_score", 0.0))

        if clip_breed in preferred_breeds:
            return 0.5, f"razza calcolata da CLIP corrisponde ({dog.get(f'clip_breed_{i}', '')}, score CLIP={clip_score:.4f}): +0.5"

    return 0.0, "nessuna corrispondenza di razza: +0.0"


def compute_structured_score_details(dog, family):
    details = []

    details.append(("età", match_preference(family["preferred_age"], dog["age_group"]), 1,
                    f"preferenza={family['preferred_age']}, cane={dog['age_group']}"))

    details.append(("taglia", size_similarity(family["preferred_size"], dog["maturity_size_label"]), 1,
                    f"preferenza={family['preferred_size']}, cane={dog['maturity_size_label']}"))

    details.append(("pelo", match_preference(family["preferred_fur"], dog["fur_length_label"]), 1,
                    f"preferenza={family['preferred_fur']}, cane={dog['fur_length_label']}"))

    breed_score, breed_explanation = compute_breed_score(dog, family)

    if breed_score is not None:
        details.append(("razza", breed_score, 3, breed_explanation))

    if family["children"]:
        children_score = 1.0 if dog["age_group"] in ["puppy", "young"] else 0.5
        details.append(("bambini", children_score, 1, f"bambini presenti, cane={dog['age_group']}"))

    if dog["maturity_size_label"] == "large":
        garden_score = 1.0 if family["garden"] else 0.2
        details.append(("giardino", garden_score, 1, f"cane large, garden={family['garden']}"))
    else:
        details.append(("giardino", 1.0, 1, "non large oppure nessuna penalizzazione"))

    if family["daily_time"] == "1-2":
        time_score = 0.6 if dog["age_group"] == "puppy" else 0.8
        details.append(("tempo disponibile", time_score, 1, f"daily_time=1-2, cane={dog['age_group']}"))
    else:
        details.append(("tempo disponibile", 1.0, 1, f"daily_time={family['daily_time']}"))

    if family["dog_experience"] == "none":
        if dog["maturity_size_label"] == "large":
            experience_score = 0.2
        elif dog["age_group"] == "puppy":
            experience_score = 0.5
        else:
            experience_score = 1.0
    else:
        experience_score = 1.0

    details.append(("esperienza", experience_score, 1, f"esperienza={family['dog_experience']}"))

    if family["applicant_age"] == "over_60":
        senior_score = 1.0 if dog["age_group"] in ["adult", "senior"] else 0.4
        details.append(("richiedente over 60", senior_score, 1, f"cane={dog['age_group']}"))

    total_score = sum(score * weight for _, score, weight, _ in details)
    total_weight = sum(weight for _, _, weight, _ in details)

    return total_score / total_weight, details


## 9. Funzioni di ranking

In [ ]:
def compute_fast_structured_ranking(family, top_n=100):
    candidates, block_reason = apply_hard_filters(dogs, family)

    if len(candidates) == 0:
        return candidates, block_reason

    structured_scores = []
    breed_scores = []
    breed_explanations = []

    for _, row in candidates.iterrows():
        structured_score, details = compute_structured_score_details(row, family)
        structured_scores.append(structured_score)

        breed_detail = [d for d in details if d[0] == "razza"]

        if len(breed_detail) > 0:
            breed_scores.append(breed_detail[0][1])
            breed_explanations.append(breed_detail[0][3])
        else:
            breed_scores.append(np.nan)
            breed_explanations.append("razza ignorata")

    candidates["breed_score"] = breed_scores
    candidates["breed_explanation"] = breed_explanations
    candidates["structured_score"] = structured_scores

    candidates = candidates.sort_values("structured_score", ascending=False).reset_index(drop=True)

    return candidates.head(top_n).copy(), ""


def compute_final_ranking_for_top_candidates(top_candidates, family):
    if len(top_candidates) == 0:
        return top_candidates

    result = top_candidates.copy()

    family_bert_embedding = get_bert_embedding(family["ideal_dog_description"])

    candidate_indices = result["original_index"].astype(int).values
    candidate_bert_embeddings = bert_embeddings[candidate_indices]

    bert_scores = cosine_similarity(
        family_bert_embedding.reshape(1, -1),
        candidate_bert_embeddings
    ).flatten()

    result["bert_similarity"] = (bert_scores + 1) / 2

    family_clip_text_feature = get_clip_text_feature(family["ideal_dog_description"])

    clip_image_text_scores = []
    clip_num_images_used = []

    for _, row in tqdm(result.iterrows(), total=len(result)):
        image_feature, num_images_used = get_clip_mean_image_feature_for_pet(row["PetID"])

        if image_feature is None:
            clip_image_text_scores.append(0.0)
            clip_num_images_used.append(0)
        else:
            with torch.no_grad():
                similarity = (image_feature @ family_clip_text_feature.T).item()

            normalized_similarity = (similarity + 1) / 2
            clip_image_text_scores.append(normalized_similarity)
            clip_num_images_used.append(num_images_used)

    result["clip_image_text_similarity"] = clip_image_text_scores
    result["clip_num_images_used_for_similarity"] = clip_num_images_used

    result["final_score"] = (
        0.6 * result["structured_score"] +
        0.2 * result["bert_similarity"] +
        0.2 * result["clip_image_text_similarity"]
    )

    return result.sort_values("final_score", ascending=False).reset_index(drop=True)


def run_case(profile_id, family, top_n_initial=100, top_n_show=20):
    print("=" * 100)
    print("CASO:", family["profile_name"])
    print("ID:", profile_id)
    print("=" * 100)

    print("Profilo famiglia")
    display(pd.DataFrame([family]).T.rename(columns={0: "valore"}))

    top_candidates, block_reason = compute_fast_structured_ranking(family, top_n=top_n_initial)

    if len(top_candidates) == 0:
        print("Nessun candidato.")
        print("Motivo:", block_reason)
        return pd.DataFrame()

    print(f"Top {top_n_initial} candidati dopo filtri + structured score")
    display(top_candidates[[
        "Name", "PetID", "Age", "age_group", "gender_label",
        "breed1_label", "breed2_label",
        "clip_breed_1", "maturity_size_label",
        "breed_score", "structured_score"
    ]].head(top_n_show))

    ranking = compute_final_ranking_for_top_candidates(top_candidates, family)

    ranking_columns = [
        "Name", "PetID", "Age", "age_group",
        "gender_label",
        "breed1_label", "breed2_label",
        "clip_breed_1", "clip_breed_1_score",
        "clip_breed_2", "clip_breed_2_score",
        "maturity_size_label",
        "breed_score", "breed_explanation",
        "structured_score",
        "bert_similarity",
        "clip_image_text_similarity",
        "final_score",
        "clip_num_images_used_for_similarity"
    ]

    print("Ranking finale")
    display(ranking[ranking_columns].head(top_n_show))

    winner = ranking.iloc[0]

    print("CANE VINCITORE")
    print("-" * 80)
    print("Nome:", winner["Name"])
    print("PetID:", winner["PetID"])
    print("Età:", winner["Age"], "mesi")
    print("Fascia età:", winner["age_group"])
    print("Sesso:", winner["gender_label"])
    print("Razza primaria dataset:", winner["breed1_label"])
    print("Razza secondaria dataset:", winner["breed2_label"])
    print("CLIP breed 1:", winner["clip_breed_1"], "-", round(winner["clip_breed_1_score"], 4))
    print("CLIP breed 2:", winner["clip_breed_2"], "-", round(winner["clip_breed_2_score"], 4))
    print("Structured score:", round(winner["structured_score"], 4))
    print("BERT similarity:", round(winner["bert_similarity"], 4))
    print("CLIP image-text similarity:", round(winner["clip_image_text_similarity"], 4))
    print("Final score:", round(winner["final_score"], 4))

    print()
    print("TUTTE LE FOTO DEL CANE VINCITORE")
    display_all_dog_images(winner["PetID"])

    structured_score, details = compute_structured_score_details(winner, family)
    details_df = pd.DataFrame(details, columns=["componente", "punteggio", "peso", "spiegazione"])
    details_df["contributo_pesato"] = details_df["punteggio"] * details_df["peso"]

    print("Dettaglio structured score")
    display(details_df)

    ranking["profile_id"] = profile_id
    ranking["profile_name"] = family["profile_name"]

    return ranking[["profile_id", "profile_name"] + ranking_columns]


## 10. Esecuzione su tutti i profili famiglia del JSON

In [ ]:
all_rankings = {}

for profile_id, family in family_profiles.items():
    ranking = run_case(
        profile_id=profile_id,
        family=family,
        top_n_initial=100,
        top_n_show=20
    )
    all_rankings[profile_id] = ranking


## 11. Salvataggio dei ranking

In [ ]:
non_empty_rankings = [
    ranking for ranking in all_rankings.values()
    if ranking is not None and len(ranking) > 0
]

if len(non_empty_rankings) > 0:
    all_rankings_df = pd.concat(non_empty_rankings, ignore_index=True)

    output_path = results_dir / "rankings_with_clip_optimized_json_families_with_winner_photos.csv"
    all_rankings_df.to_csv(output_path, index=False)

    print("Ranking salvati in:", output_path)
    print("Dimensione:", all_rankings_df.shape)

    display(all_rankings_df.head(30))
else:
    print("Nessun ranking da salvare.")


## Conclusione

Il notebook esegue il matching su più famiglie caricate da JSON.

Per ogni famiglia mostra:

- profilo famiglia;
- Top 100 candidati dopo filtri e structured score;
- ranking finale;
- cane vincitore;
- **tutte le foto del cane vincitore**;
- dettaglio del punteggio strutturato.

Formula finale:

```text
final_score =
0.6 * structured_score +
0.2 * bert_similarity +
0.2 * clip_image_text_similarity
```
